# HAI817 - Machine Learning  
### Champenois Mathys - Czolacz Mickael - Gellida Hugo

## Installation

In [1]:
!pip install csv
!pip install numpy
!pip install matplotlib

ERROR: Could not find a version that satisfies the requirement csv (from versions: none)
ERROR: No matching distribution found for csv


In [23]:
# Importation des différentes librairies, classes et fonctions utiles

# Suppression des warnings de Scikit-Learn liés aux futures versions
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Librairies générales
import pandas as pd  # Manipulation de données sous forme de DataFrame
import re  # Manipulation des chaînes de caractères avec expressions régulières
import time  # Mesure du temps d'exécution
import numpy as np  # Calcul scientifique et manipulation de tableaux
import pickle  # Sauvegarde et chargement d'objets Python
import sys  # Gestion des fonctions et paramètres système

# Librairies pour l'affichage
import matplotlib.pyplot as plt  # Création de graphiques statiques
import seaborn as sns  # Visualisation avancée basée sur Matplotlib

# Librairies Scikit-Learn pour la vectorisation et les pipelines

from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)

from sklearn.base import BaseEstimator, TransformerMixin # Estimator personels
from sklearn.pipeline import Pipeline                          # Pipeline

from sklearn.model_selection import train_test_split  # Split jeu de données
from sklearn.model_selection import cross_val_score        # Validation croisee
from sklearn.model_selection import KFold, GridSearchCV    # KFold + grid

from sklearn import metrics                       # Metriques
from sklearn.metrics import (confusion_matrix,
                             classification_report,
                             accuracy_score)
from sklearn.decomposition import PCA     # Réduction de dimension (ACP)
from mpl_toolkits.mplot3d import Axes3D  # nécessaire pour la 3D

# Classifiers utilisés dans le notebook
from sklearn.svm import SVC  # Support Vector Machine

# Librairies NLTK pour le traitement du langage naturel
import nltk  # Librairie principale de NLP
from nltk.stem import WordNetLemmatizer  # Lemmatisation des termes
from nltk.stem import PorterStemmer  # Racinisation des termes
from nltk.corpus import stopwords  # Liste des stopwords
from nltk import word_tokenize  # Découpage en tokens

# Téléchargement des ressources nécessaires de NLTK
nltk.download('wordnet')  # Pour la lemmatisation
nltk.download('stopwords')  # Liste des stopwords
nltk.download('punkt_tab')  # Tokenisation de phrases et mots

# Définition des stopwords anglais
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\3nd3r\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\3nd3r\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\3nd3r\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
#Ouverture fichier tsv
bdd = pd.read_csv("scitweets_export.tsv", sep='\t')

print(bdd.loc[bdd.scientific_context == 1.0])

      Unnamed: 0             tweet_id  \
6              8   333266791960809472   
8             10   334350038090276864   
14            16   344256820552011777   
32            35   363172565365170176   
38            41   371197661439078400   
...          ...                  ...   
1126        1244  1335031921626279937   
1128        1247  1336903053362954241   
1130        1249  1338450113765789699   
1131        1250  1338483958397480972   
1134        1253  1339398314563792900   

                                                   text  science_related  \
6     “Traffic Jam” In Brain’s Neurons Could Be Caus...                1   
8     The effect of climate change on iceberg produc...                1   
14    @RepCohen @SenAlexander @SenBobCorker pls supp...                1   
32    MDLinx Blog: MDLinx Blog: Neuropsychological f...                1   
38    @najdi_outsider @mayami2012 @mol7daa @ahlamm4 ...                1   
...                                                

Cleanup, on retire la colonne Unnamed:0, une erreur d'import?

In [8]:
print(bdd.columns)
df = bdd.copy()
df = df.drop(columns=['Unnamed: 0'])
print(df)
print(df.columns)

Index(['Unnamed: 0', 'tweet_id', 'text', 'science_related', 'scientific_claim',
       'scientific_reference', 'scientific_context'],
      dtype='object')
                 tweet_id                                               text  \
0      316669998137483264  Knees are a bit sore. i guess that's a sign th...   
1      319090866545385472          McDonald's breakfast stop then the gym 🏀💪   
2      322030931022065664  Can any Gynecologist with Cancer Experience ex...   
3      322694830620807168  Couch-lock highs lead to sleeping in the couch...   
4      328524426658328576  Does daily routine help prevent problems with ...   
...                   ...                                                ...   
1135  1340455669443350528  @LaylaFanucci @realDonaldTrump I'm sorry but w...   
1136  1340689510569549824  Dear #NIN applicants, you can kindly download ...   
1137  1341155832793165825         Whats the uber support team email address?   
1138  1344167355648241664  House passes bill

# vectorisation

## écriture dans un df['text_vectorized', science_related]

Features : ['000' '01' '049' ... '音樂' '더쇼' '런쥔을_공평하게_대하세요']
Tout est converti en minuscules

Vocabulaire :
{'knees': 3908, 'are': 665, 'bit': 997, 'sore': 6521, 'guess': 3096, 'that': 6958, 'sign': 6399, 'my': 4682, 'recent': 5786, 'treadmilling': 7168, 'is': 3668, 'working': 7750, 'mcdonald': 4373, 'breakfast': 1102, 'stop': 6658, 'then': 6981, 'the': 6960, 'gym': 3117, 'can': 1231, 'any': 624, 'gynecologist': 3118, 'with': 7718, 'cancer': 1236, 'experience': 2541, 'explain': 2548, 'dangers': 1869, 'of': 4967, 'transvaginal': 7162, 'douching': 2185, 'fluoride': 2764, 'or': 5060, 'other': 5082, 'toxins': 7131, 'such': 6734, 'as': 699, 'dioxin': 2070, 'pdx': 5214, 'couch': 1723, 'lock': 4152, 'highs': 3260, 'lead': 4021, 'to': 7083, 'sleeping': 6447, 'in': 3505, 'gotta': 3034, 'doing': 2157, 'this': 7009, 'shit': 6357, 'does': 2148, 'daily': 1856, 'routine': 6062, 'help': 3225, 'prevent': 5484, 'problems': 5509, 'bipolar': 990, 'disorder': 2111, 'http': 3363, 'co': 1508, 'xgufudoljb': 7

CountVectorizer(lowercase=False)


In [16]:
df_sci = df.query("science_related==1")
df_sci

,tweet_id,text,science_related,scientific_claim,scientific_reference,scientific_context
2,322030931022065664,Can any Gynecologist with Cancer Experience ex...,1,1.0,0.0,0.0
3,322694830620807168,Couch-lock highs lead to sleeping in the couch...,1,1.0,0.0,0.0
4,328524426658328576,Does daily routine help prevent problems with ...,1,1.0,0.0,0.0
6,333266791960809472,“Traffic Jam” In Brain’s Neurons Could Be Caus...,1,1.0,1.0,1.0
7,334282732085587968,Can playing more games improve lives and save ...,1,1.0,0.0,0.0
...,...,...,...,...,...,...
1128,1336903053362954241,Three systematic reviews & the WHO contradict ...,1,0.0,1.0,1.0
1130,1338450113765789699,This looks like a great opportunity to get res...,1,0.0,0.0,1.0
1131,1338483958397480972,Highly prestigious and competitive awards fund...,1,0.0,0.0,1.0
1134,1339398314563792900,"Vestislav Apostolov, David M. J. Calderbank, E...",1,0.0,1.0,1.0


In [17]:
df_nsci = df.query("science_related==0")
df_nsci

,tweet_id,text,science_related,scientific_claim,scientific_reference,scientific_context
0,316669998137483264,Knees are a bit sore. i guess that's a sign th...,0,0.0,0.0,0.0
1,319090866545385472,McDonald's breakfast stop then the gym 🏀💪,0,0.0,0.0,0.0
5,331396203700944896,The Impact of Infertility on You and Your Rela...,0,0.0,0.0,0.0
9,335542235452014594,@HankAzaria @TheSimpsons oh loneliness and che...,0,0.0,0.0,0.0
10,336431788618551297,#bbcsportsday Honestly think Liverpool are a l...,0,0.0,0.0,0.0
...,...,...,...,...,...,...
1133,1339272384763801605,@Madz_Grant Wroter for a facist rag supports t...,0,0.0,0.0,0.0
1136,1340689510569549824,"Dear #NIN applicants, you can kindly download ...",0,0.0,0.0,0.0
1137,1341155832793165825,Whats the uber support team email address?,0,0.0,0.0,0.0
1138,1344167355648241664,House passes bill to increase stimulus checks ...,0,0.0,0.0,0.0


On cherche un bon classifier :

In [ ]:
clfs = [KNeighborsClassifier(), DecisionTreeClassifier(), GaussianNB(), SVC(), RandomForestClassifier()]
kfold = KFold(10)
scoring = 'accuracy'
results = []
names = []
for clf in clfs:
    score_kfold = cross_val_score(clf, X_train, y_train, cv=kfold, scoring=scoring)
    results.append(score_kfold)
    names.append(clf.__class__.__name__)
    print(f"{clf.__class__.__name__} : Moyenne : {np.mean(score_kfold):.3f}, écart-type : {np.std(score_kfold):.3f}")


## Explication des données:

- tweet_id : identification du tweet
- text : contenu du tweet
- science_related : article relié à la science
- scientific_claim : article faisant une affirmation ou une hypothèse scientifique
- scientific_reference : article citant une référence scientifique
- scientific_context : article faisant partie d'une discussion scientifique, et non un article personnel